# Step 3 — On-Device Speech Transcription
**Tough Talks · Phase 2**

Goal: prove out **Gemma 4 E2B's native audio path** end-to-end,
**including audio longer than Gemma 4's 30 s window** (Live Mode will see
multi-minute conversations).

We reuse the same multimodal model that will power emotion + reasoning so the
Live-Mode stack stays single-model. Notebook is a thin driver — all logic in
`backend/core/_runtime/audio.py`.

**Test audio — license note for Kaggle**
The clip is synthesised in-notebook with **gTTS** (MIT-licensed) from a custom
nine-turn argument script we wrote for Tough Talks. No third-party audio is
downloaded, so there is no upstream license to track or attribute. Two voices
come from gTTS region accents (`tld='com'` and `tld='co.uk'`).

**What "done" looks like for this step**
1. Multimodal Gemma 4 loads (`AutoModelForMultimodalLM` via `LoadConfig(multimodal=True)`).
2. The synthesised argument WAV is **> 30 s** (Gemma 4's per-call cap).
3. `transcribe_long()` chunks it into ≤ 28 s windows, transcribes each,
   stitches the transcripts, and emits one schema-conforming dict whose
   `segments` field records per-window boundaries.
4. The result dict validates against `data/schemas/transcription.schema.json`.

**Gemma 4 audio constraints worth remembering**
- Audio content must come **before** the text instruction in `content`.
- Per-call max is **30 seconds** (`MAX_AUDIO_SECONDS`). Longer clips are
  chunked at `DEFAULT_CHUNK_SECONDS` (28 s) with a small overlap.
- Audio support is on **E2B and E4B only** (other Gemma 4 sizes are text/vision).

In [1]:
# ── 0. Install / upgrade dependencies ────────────────────────────────────────
# Phase 2 adds:
#   - librosa + soundfile : audio feature extractor + I/O for Gemma 4 audio
#   - gTTS                : MIT-licensed text-to-speech to synthesise the test
#                           argument clip in-notebook (no third-party audio
#                           to license-track for the Kaggle submission)
#
# DO NOT bump torch on Colab/Kaggle — it breaks the pre-installed
# torchvision/CUDA pairing. Only bump transformers + accelerate + audio libs.
# After this cell runs once, RESTART THE KERNEL before re-running anything,
# otherwise the already-imported transformers module won't pick up the upgrade.

!pip install -q -U transformers accelerate librosa soundfile gTTS

In [2]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ───────────────────────
# Same shim as Step 1 — auto-clones / refreshes on Colab / Kaggle. After
# refreshing the working tree we also drop any cached `backend.*` modules
# from sys.modules so subsequent `from backend.core._runtime import X`
# picks up the freshly-pulled code instead of whatever this kernel imported
# earlier in the session.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Drop cached backend.* modules so later imports re-read from the
# just-refreshed working tree (avoids stale exports across kernel-alive runs).
_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

Refreshing /content/tough_talks from origin
Repo root: /content/tough_talks


In [3]:
# ── 2. Imports ───────────────────────────────────────────────────────────────
import io
import json
from pathlib import Path

import numpy as np
import torch

from backend.core._runtime import (
    DEFAULT_CHUNK_SECONDS,
    DEFAULT_MODEL_ID,
    LoadConfig,
    MAX_AUDIO_SECONDS,
    TranscribeConfig,
    load_model,
    transcribe_long,
)

In [4]:
# ── 3. Configuration ─────────────────────────────────────────────────────────

MODEL_ID = DEFAULT_MODEL_ID                                # google/gemma-4-E2B-it
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

# Custom Tough Talks-style argument script. Nine turns alternating between two
# speakers (US vs UK gTTS accents). The arc: missed deadline → blame → mutual
# escalation → de-escalation → resolution. Sized to land at ~40–45 s so the
# clip deliberately exceeds Gemma 4's 30 s audio cap — that's what
# transcribe_long() is here to handle.
ARGUMENT_SCRIPT = [
    ("us", "I asked for the report on Monday and it's already Thursday. What happened?"),
    ("uk", "I told you on Tuesday the data team hadn't delivered. I can't make numbers up."),
    ("us", "Then you escalate. You don't just sit on it. The whole quarter close depends on this."),
    ("uk", "I did escalate. You weren't in the meeting. Don't blame me for your missed message."),
    ("us", "Look — I can't be in every meeting. That's why we have email."),
    ("uk", "I sent two emails. You replied to neither. Don't put this on me."),
    ("us", "Okay, fair. I missed them. But we still have a problem to solve tonight."),
    ("uk", "I have partial numbers from the staging tables. We can present those and flag the gaps."),
    ("us", "Good. Let's regroup at six. And next time, just call me directly."),
]

TARGET_SR    = 16000  # Gemma 4's audio extractor resamples to 16 kHz anyway
SILENCE_S    = 0.25   # short gap between turns to make the dialogue parse-able
TLD_MAP      = {"us": "com", "uk": "co.uk"}

AUDIO_DIR    = Path(REPO_ROOT) / "data" / "audio_cache"
AUDIO_LOCAL  = AUDIO_DIR / "argument_synth.wav"

SCHEMA_PATH  = Path(REPO_ROOT) / "data" / "schemas" / "transcription.schema.json"

print(f"Model       : {MODEL_ID}")
print(f"Device      : {DEVICE}")
print(f"Audio       : {AUDIO_LOCAL}")
print(f"Per-call cap: {MAX_AUDIO_SECONDS}s   chunk size: {DEFAULT_CHUNK_SECONDS}s")
print(f"Script      : {len(ARGUMENT_SCRIPT)} turns, ~{sum(len(t.split()) for _, t in ARGUMENT_SCRIPT)} words")

Model       : google/gemma-4-E2B-it
Device      : cuda
Audio       : /content/tough_talks/data/audio_cache/argument_synth.wav
Per-call cap: 30s   chunk size: 28.0s
Script      : 9 turns, ~127 words


In [5]:
# ── 4. Synthesise the argument clip (gitignored) ─────────────────────────────
# Per-turn rendering: write each line to an MP3 buffer via gTTS, load it
# through librosa (resamples to 16 kHz mono), then concatenate with a short
# silence between turns. Result is one WAV that lives in
# data/audio_cache/ (gitignored via the *.wav + audio_cache/ rules).
#
# Idempotent: skip the render if the WAV is already cached.
# NOTE: we intentionally target a duration > 30 s to force the chunked path
# in cell 6 — that's the whole point of the step.

import librosa
import soundfile as sf
from gtts import gTTS

AUDIO_DIR.mkdir(parents=True, exist_ok=True)

if AUDIO_LOCAL.exists():
    print(f"Argument clip already cached: {AUDIO_LOCAL} ({AUDIO_LOCAL.stat().st_size/1024:.1f} KB)")
else:
    print("Synthesising argument clip via gTTS ...")
    chunks: list[np.ndarray] = []
    silence = np.zeros(int(TARGET_SR * SILENCE_S), dtype=np.float32)
    for i, (accent, line) in enumerate(ARGUMENT_SCRIPT, start=1):
        buf = io.BytesIO()
        gTTS(text=line, lang="en", tld=TLD_MAP[accent]).write_to_fp(buf)
        buf.seek(0)
        wave, _ = librosa.load(buf, sr=TARGET_SR, mono=True)
        chunks.append(wave.astype(np.float32))
        chunks.append(silence)
        print(f"  turn {i} [{accent}] {len(wave)/TARGET_SR:.2f}s — {line[:60]}...")
    combined = np.concatenate(chunks)
    sf.write(str(AUDIO_LOCAL), combined, TARGET_SR)
    print(f"Saved {AUDIO_LOCAL} ({AUDIO_LOCAL.stat().st_size/1024:.1f} KB)")

# Duration probe — for this step we *want* > 30 s so the chunked path is
# exercised. No hard assertion either way; the results table will flag it.
info = sf.info(str(AUDIO_LOCAL))
DURATION_S = info.frames / info.samplerate
print(f"Duration: {DURATION_S:.2f}s @ {info.samplerate} Hz, {info.channels}ch")
if DURATION_S > MAX_AUDIO_SECONDS:
    print(f"  → exceeds Gemma 4's {MAX_AUDIO_SECONDS}s per-call cap — transcribe_long() will chunk this.")
else:
    print(f"  → fits in a single Gemma 4 audio window; transcribe_long() will run one pass.")

Synthesising argument clip via gTTS ...
  turn 1 [us] 4.92s — I asked for the report on Monday and it's already Thursday. ...
  turn 2 [uk] 5.52s — I told you on Tuesday the data team hadn't delivered. I can'...
  turn 3 [us] 6.00s — Then you escalate. You don't just sit on it. The whole quart...
  turn 4 [uk] 6.24s — I did escalate. You weren't in the meeting. Don't blame me f...
  turn 5 [us] 4.42s — Look — I can't be in every meeting. That's why we have email...
  turn 6 [uk] 6.12s — I sent two emails. You replied to neither. Don't put this on...
  turn 7 [us] 5.45s — Okay, fair. I missed them. But we still have a problem to so...
  turn 8 [uk] 6.24s — I have partial numbers from the staging tables. We can prese...
  turn 9 [us] 5.50s — Good. Let's regroup at six. And next time, just call me dire...
Saved /content/tough_talks/data/audio_cache/argument_synth.wav (1645.4 KB)
Duration: 52.65s @ 16000 Hz, 1ch
  → exceeds Gemma 4's 30s per-call cap — transcribe_long() will chunk this.


In [6]:
# ── 5. Load the multimodal processor + model ─────────────────────────────────
# multimodal=True swaps AutoModelForCausalLM for AutoModelForMultimodalLM so
# the audio inputs in apply_chat_template() are actually consumed by the model.

processor, model = load_model(LoadConfig(model_id=MODEL_ID, multimodal=True))

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded ({n_params:.1f}B parameters, on {model.device})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
[transformers] Current model requires 6178 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Model loaded (5.1B parameters, on cpu)


In [7]:
# ── 6. Transcribe (chunked path handles >30s) ────────────────────────────────
# transcribe_long() splits clips longer than DEFAULT_CHUNK_SECONDS (28s) into
# overlapping windows, runs transcribe() on each, and stitches the results.
# For clips already short enough it delegates straight to transcribe() and
# returns no segments field.

cfg = TranscribeConfig(
    max_new_tokens=512,
    language="en",
    duration_seconds=round(DURATION_S, 2),
)

result = transcribe_long(processor, model, AUDIO_LOCAL, cfg=cfg, model_id=MODEL_ID)

# Suppress the long instruction string from the displayed output — it's the
# same default for every call and adds noise.
display_result = {k: v for k, v in result.items() if k != "instruction"}
print(json.dumps(display_result, indent=2))

{
  "audio_source": "/content/tough_talks/data/audio_cache/argument_synth.wav",
  "transcript": "I asked for the report on Monday and it's already Thursday. What happened? I told you on Tuesday the data team hadn't delivered. I can't make numbers up. Then you escalate. You don't just sit on it. The whole quarter close depends on this. I did escalate. You weren't in the meeting. Don't blame me for your missed message. Look, I can't be in every meeting. That's why we have email. I sent two emails. You replied to neither. Don't put this on me. Okay, fair. I missed them, but we still have a problem to solve tonight. I have partial numbers from the staging tables. We can present those and flag the gaps. Good. Let's regroup at six. And next time, just call me directly.",
  "language": "en",
  "duration_seconds": 52.65,
  "model_id": "google/gemma-4-E2B-it",
  "created_at": "2026-05-11T10:18:32.485322+00:00",
  "segments": [
    {
      "start_seconds": 0.0,
      "end_seconds": 28.0,
      "

In [8]:
# ── 7. Schema validation (no third-party deps) ───────────────────────────────
# Hand-checks the contract instead of pulling jsonschema in. Covers:
#   * required keys present
#   * field types match
#   * no surprise top-level keys (additionalProperties: false)

schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))

_TYPE_MAP = {
    "string": (str,),
    "number": (int, float),
    "boolean": (bool,),
    "integer": (int,),
    "object": (dict,),
    "array": (list,),
    "null": (type(None),),
}

def _validate_against_schema(payload: dict, schema: dict) -> list[str]:
    errors: list[str] = []
    required = schema.get("required", [])
    props = schema.get("properties", {})
    for key in required:
        if key not in payload:
            errors.append(f"missing required field: {key!r}")
    for key, value in payload.items():
        if key not in props:
            if schema.get("additionalProperties") is False:
                errors.append(f"unexpected top-level field: {key!r}")
            continue
        spec = props[key]
        declared = spec.get("type")
        types = [declared] if isinstance(declared, str) else (declared or [])
        if not types:
            continue
        allowed = tuple(t for name in types for t in _TYPE_MAP.get(name, ()))
        if allowed and not isinstance(value, allowed):
            errors.append(f"{key!r}: expected {types}, got {type(value).__name__}")
    return errors

schema_errors = _validate_against_schema(result, schema)
if schema_errors:
    print("SCHEMA ERRORS:")
    for e in schema_errors:
        print(f"  - {e}")
else:
    print("Schema validation: OK")

Schema validation: OK


In [9]:
# ── 8. Step 3 results table ──────────────────────────────────────────────────
# Renders unconditionally — same pattern as Step 1 so partial failures are
# visible at a glance.

n_segments = len(result.get("segments") or [])
if DURATION_S > DEFAULT_CHUNK_SECONDS:
    chunking_ok   = n_segments >= 2
    chunking_note = f"{n_segments} chunks for {DURATION_S:.1f}s audio"
else:
    chunking_ok   = n_segments == 0
    chunking_note = f"single window ({DURATION_S:.1f}s ≤ {DEFAULT_CHUNK_SECONDS}s)"

checks: list[tuple[str, bool, str]] = [
    ("multimodal_model_loaded", True, f"{n_params:.1f}B params on {model.device}"),
    ("audio_synthesised",       AUDIO_LOCAL.exists(), f"{DURATION_S:.2f}s"),
    ("over_30s_test_path",      DURATION_S > MAX_AUDIO_SECONDS, f"{DURATION_S:.1f}s > {MAX_AUDIO_SECONDS}s cap"),
    ("chunked_correctly",       chunking_ok, chunking_note),
    ("transcript_non_empty",    bool(result.get("transcript")), f"{len(result.get('transcript', ''))} chars"),
    ("schema_valid",            not schema_errors, "ok" if not schema_errors else f"{len(schema_errors)} error(s)"),
]

print("=" * 70)
print("STEP 3 RESULTS — Gemma 4 native speech transcription (long-audio path)")
print("=" * 70)
all_ok = True
for name, ok, note in checks:
    icon = "PASS" if ok else "FAIL"
    print(f"[{icon}]  {name:28s}  {note}")
    if not ok:
        all_ok = False

print()
print("Stitched transcript:")
print(f"  {result.get('transcript', '')}")
print()
if n_segments:
    print("Per-chunk segments:")
    for i, seg in enumerate(result["segments"], start=1):
        print(f"  [{i}] {seg['start_seconds']:5.2f}–{seg['end_seconds']:5.2f}s  "
              f"{seg['transcript'][:120]}")
    print()
print("OVERALL:", "READY FOR STEP 4" if all_ok else "FIX FAILURES ABOVE")

STEP 3 RESULTS — Gemma 4 native speech transcription (long-audio path)
[PASS]  multimodal_model_loaded       5.1B params on cpu
[PASS]  audio_synthesised             52.65s
[PASS]  over_30s_test_path            52.6s > 30s cap
[PASS]  chunked_correctly             2 chunks for 52.6s audio
[PASS]  transcript_non_empty          676 chars
[PASS]  schema_valid                  ok

Stitched transcript:
  I asked for the report on Monday and it's already Thursday. What happened? I told you on Tuesday the data team hadn't delivered. I can't make numbers up. Then you escalate. You don't just sit on it. The whole quarter close depends on this. I did escalate. You weren't in the meeting. Don't blame me for your missed message. Look, I can't be in every meeting. That's why we have email. I sent two emails. You replied to neither. Don't put this on me. Okay, fair. I missed them, but we still have a problem to solve tonight. I have partial numbers from the staging tables. We can present those and f